# [초격차] AI 헬스케어 3기 해커톤: 난임 환자 임신 성공 여부 예측

**평가지표:** ROC-AUC  
**전략:** Phase 1 (XGB×Cat Optuna 30회) → Phase 2 (트리 다양성) → Phase 3 (TabNet/FT-Transformer) → Phase 4 (2단계 스태킹)  
**Data Leakage 원칙:** train에서만 fit, test는 transform only


# 실험 로그

| 실험 | 모델 | Feature Engineering | 파라미터 | Score | 비고 | 리더보드 점수 |
|------|------|---------------------|----------|-------|------|------|
| 4/24(금)|
| exp0 | LightGBM | 기본 feature + 파생변수 110개 | C=1.0, max_iter=100 | 0.7385 | | 리더보드 점수 : 0.5 |
| exp1 | XGBoost | 기본 feature + 파생변수 110개 | lr=0.05, depth=6, subsample=0.8 | 0.7392 | | 리더보드 점수 : 0.5|
| exp2 | Catboost | - | iterations=2000, lr=0.05, depth=6 | 0.7397 | | 리더보드 점수 : 0.7412 |
| exp3 | XGBoost x Catboost 앙상블 | 기본 feature + 파생변수 110개 | iterations=2000, lr=0.05, depth=6  | 0.7399 | A. Simple Average (XGB : Cat = 1:1) |
| exp4 | - |  -  |  - | 0.7399 | B. Weighted Average(OOF AUC가 높은 모델 가중치 부여) |
| exp5 | - |  -  |  - | 0.7399 | C. Stacking (메타 모델 : LR) |
| exp6 | - |  -  |  - | 0.7399 | D. Rank Average(확률 대신 순위로 변환 후 평균) |
| exp7 | XGBoost | 기본 feature + 파생변수 110개 | XGB: depth=5 / Cat: depth=5  | 0.7397 |optuna|
| exp8 | Catboost | - | XGB: depth=5 / Cat: depth=5  | 0.7397 |optuna |
| exp7 | XGBoost x Catboost 앙상블(optuna) |  기본 feature + 파생변수 110개  | XGB: depth=5 / Cat: depth=5 | 0.7399 | A. Simple Average (XGB : Cat = 1:1) |
| exp8 | - |  -  |  - | 0.7399 | B. Weighted Average(OOF AUC가 높은 모델 가중치 부여) |
| exp9 | - |  -  |  - | 0.7399 | C. Stacking (메타 모델 : LR) |
| exp10 | - |  -  |  - | 0.7399 | D. Rank Average(확률 대신 순위로 변환 후 평균) | 리더보드 점수 : 0.74162 |
| 4/25(토) |
| exp11 | LGBM x XGBoost x Catboost 앙상블(optuna) |  기본 feature + 파생변수 110개  | LGBM : max_depth=3 / XGB: max_depth=5 / Cat: depth=6 | 0.7400 | |리더보드 점수 :  0.74140  |
| 4/26(일)|
| exp12 | XGBoost + CatBoost + LightGBM + RandomForest + ExtraTree 앙상블(optuna) |  기본 feature + 파생변수 110개  | LGBM : max_depth=3 / XGB: max_depth=5 / Cat: depth=6 | 0.7398 | Stacking (메타모델 : LR) |
| exp13 | XGBoost + CatBoost + LightGBM 앙상블(파생변수 추가) |  기본 feature + 파생변수 110개 + 신규 파생변수 29  | - | 0.73788 | |
| exp14 | seed 앙상블(파생변수 추가) |  기본 feature + 파생변수 110개 + 신규 파생변수 29  | - | 0.73998 | | 리더보드 점수 : 0.74182 |

## 0. 패키지 설치 및 임포트

In [1]:
! pip install -q catboost lightgbm xgboost optuna pytorch-tabnet
! pip install -q koreanize-matplotlib
! pip install pytorch-tabnet

import warnings, os, time
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from scipy.optimize import minimize

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    import koreanize_matplotlib
except:
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

RANDOM_STATE = 42
N_SPLITS     = 5
N_OPTUNA     = 30        # Optuna 탐색 횟수
np.random.seed(RANDOM_STATE)

TARGET   = "임신 성공 여부"
ID_COL   = "ID"
DATA_DIR = Path(".")

train = pd.read_csv('../data/train.csv', encoding="utf-8-sig")
test  = pd.read_csv('../data/test.csv',  encoding="utf-8-sig")
sub = pd.read_csv('../data/sample_submission.csv', encoding="utf-8-sig")
train.columns = [c.strip() for c in train.columns]
test.columns  = [c.strip() for c in test.columns]

print(f"Train: {train.shape} | Test: {test.shape}")
print(f"양성 비율: {train[TARGET].mean():.4f}  (불균형: {(train[TARGET]==0).sum()}:{(train[TARGET]==1).sum()})")



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Train: (256351, 69) | Test: (90067, 68)
양성 비율: 0.2583  (불균형: 190123:66228)


## 1. 인코딩 맵 & 전처리 상수

In [3]:
AGE_MAP = {"만18-34세":0,"만35-37세":1,"만38-39세":2,
           "만40-42세":3,"만43-44세":4,"만45-50세":5,"알 수 없음":-1}
COUNT_MAP = {"0회":0,"1회":1,"2회":2,"3회":3,"4회":4,"5회":5,"6회 이상":6}
DONOR_AGE_MAP = {"만20세 이하":0,"만21-25세":1,"만26-30세":2,
                 "만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1}
INDUCTION_MAP = {"알 수 없음":0,"기록되지 않은 시행":1,
                 "생식선 자극 호르몬":2,"세트로타이드 (억제제)":3}
EGG_MAP   = {"본인 제공":0,"기증 제공":1,"알 수 없음":-1}
SPERM_MAP = {"배우자 제공":0,"기증 제공":1,"배우자 및 기증 제공":2,"미할당":-1}
CODE_MAP  = {v:i for i,v in enumerate(
    ["TRCMWS","TRDQAZ","TRJXFG","TRVNRY","TRXQMD","TRYBLT","TRZKPL"])}

COUNT_COLS = ["총 시술 횟수","클리닉 내 총 시술 횟수","IVF 시술 횟수","DI 시술 횟수",
              "총 임신 횟수","IVF 임신 횟수","DI 임신 횟수",
              "총 출산 횟수","IVF 출산 횟수","DI 출산 횟수"]
DROP_COLS  = ["착상 전 유전 검사 사용 여부","PGD 시술 여부","PGS 시술 여부",
              "불임 원인 - 여성 요인","난자 채취 경과일","난자 해동 경과일"]
REASON_CATS  = ["현재 시술용","배아 저장용","난자 저장용","기증용","연구용"]
PROC_TYPES   = ["IVF","ICSI","IUI","ICI","GIFT","FER",
                "BLASTOCYST","AH","Generic DI","IVI"]
print("인코딩 맵 설정 완료")


인코딩 맵 설정 완료


## 2. 전처리 함수 (Leakage-free)

In [4]:
def expand_reason(df):
    for cat in REASON_CATS:
        df[f"이유_{cat}"] = df["배아 생성 주요 이유"].fillna("").str.contains(cat).astype(int)
    return df

def expand_procedure(df):
    filled = df["특정 시술 유형"].fillna("Unknown")
    for pt in PROC_TYPES:
        df[f"시술_{pt}"] = (filled.str.upper()
                            .str.replace(" ","",regex=False)
                            .str.contains(pt.upper()).astype(int))
    return df

def preprocess(df, medians=None, fit=False):
    """fit=True → train에서 중앙값 계산 / fit=False → train 중앙값 적용"""
    df = df.copy()
    df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)

    col_years = "임신 시도 또는 마지막 임신 경과 연수"
    if col_years in df.columns:
        df["임신_시도_연수_결측"] = df[col_years].isnull().astype(int)
        df[col_years] = df[col_years].fillna(-1)

    col_thaw = "배아 해동 경과일"
    if col_thaw in df.columns:
        df["배아_해동_결측"] = df[col_thaw].isnull().astype(int)
        df[col_thaw] = df[col_thaw].fillna(0)

    fill_zero = ["단일 배아 이식 여부","착상 전 유전 진단 사용 여부","총 생성 배아 수",
                 "미세주입된 난자 수","미세주입에서 생성된 배아 수","이식된 배아 수",
                 "미세주입 배아 이식 수","저장된 배아 수","미세주입 후 저장된 배아 수",
                 "해동된 배아 수","해동 난자 수","수집된 신선 난자 수","저장된 신선 난자 수",
                 "혼합된 난자 수","파트너 정자와 혼합된 난자 수","기증자 정자와 혼합된 난자 수",
                 "동결 배아 사용 여부","신선 배아 사용 여부","기증 배아 사용 여부","대리모 여부"]
    for c in fill_zero:
        if c in df.columns: df[c] = df[c].fillna(0)

    if medians is None: medians = {}
    for col in ["난자 혼합 경과일","배아 이식 경과일"]:
        if col not in df.columns: continue
        if fit: medians[col] = df[col].median()
        df[col] = df[col].fillna(medians.get(col, 0))

    cont_cols = ["총 생성 배아 수","미세주입된 난자 수","미세주입에서 생성된 배아 수",
                 "이식된 배아 수","미세주입 배아 이식 수","저장된 배아 수",
                 "미세주입 후 저장된 배아 수","해동된 배아 수","해동 난자 수",
                 "수집된 신선 난자 수","저장된 신선 난자 수","혼합된 난자 수",
                 "파트너 정자와 혼합된 난자 수","기증자 정자와 혼합된 난자 수","배아 해동 경과일"]
    for c in cont_cols:
        if c not in df.columns: continue
        df[c] = pd.to_numeric(df[c], errors="coerce")
        if fit: medians[c] = df[c].median() if df[c].notna().any() else 0
        df[c] = df[c].fillna(medians.get(c, 0))

    if "시술 당시 나이" in df.columns:
        df["시술 당시 나이"] = df["시술 당시 나이"].map(AGE_MAP).fillna(-1).astype(int)
    for col in COUNT_COLS:
        if col in df.columns: df[col] = df[col].map(COUNT_MAP).fillna(-1).astype(int)
    for col in ["난자 기증자 나이","정자 기증자 나이"]:
        if col in df.columns: df[col] = df[col].map(DONOR_AGE_MAP).fillna(-1).astype(int)
    if "시술 유형" in df.columns:
        df["시술 유형"] = (df["시술 유형"] == "IVF").astype(int)
    if "배란 유도 유형" in df.columns:
        df["배란 유도 유형"] = df["배란 유도 유형"].map(INDUCTION_MAP).fillna(0).astype(int)
    if "난자 출처" in df.columns:
        df["난자 출처"] = df["난자 출처"].map(EGG_MAP).fillna(-1).astype(int)
    if "정자 출처" in df.columns:
        df["정자 출처"] = df["정자 출처"].map(SPERM_MAP).fillna(-1).astype(int)
    if "시술 시기 코드" in df.columns:
        df["시술 시기 코드"] = df["시술 시기 코드"].map(CODE_MAP).fillna(-1).astype(int)

    binary_cols = ["배란 자극 여부","단일 배아 이식 여부","착상 전 유전 진단 사용 여부",
                   "남성 주 불임 원인","남성 부 불임 원인","여성 주 불임 원인","여성 부 불임 원인",
                   "부부 주 불임 원인","부부 부 불임 원인","불명확 불임 원인",
                   "불임 원인 - 난관 질환","불임 원인 - 남성 요인","불임 원인 - 배란 장애",
                   "불임 원인 - 자궁경부 문제","불임 원인 - 자궁내막증",
                   "불임 원인 - 정자 농도","불임 원인 - 정자 면역학적 요인",
                   "불임 원인 - 정자 운동성","불임 원인 - 정자 형태",
                   "동결 배아 사용 여부","신선 배아 사용 여부","기증 배아 사용 여부","대리모 여부"]
    for c in binary_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    return df, medians


## 3. 파생변수 함수 (기존 46개 + 신규 29개)

In [5]:
def feature_engineering(df):
    """기존 파생변수 (문서3 기반, 32개)"""
    df = df.copy()
    for c in ["총 생성 배아 수","이식된 배아 수","저장된 배아 수","수집된 신선 난자 수",
              "미세주입에서 생성된 배아 수","미세주입된 난자 수","해동된 배아 수",
              "미세주입 후 저장된 배아 수","총 임신 횟수","총 시술 횟수",
              "클리닉 내 총 시술 횟수","IVF 시술 횟수","동결 배아 사용 여부",
              "신선 배아 사용 여부","기증 배아 사용 여부","시술 유형","시술 당시 나이"]:
        if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    cause_cols   = [c for c in df.columns if "불임 원인 -" in c]
    df["불임_원인_합계"] = df[cause_cols].sum(axis=1)
    primary_cols = [c for c in ["남성 주 불임 원인","여성 주 불임 원인","부부 주 불임 원인"] if c in df.columns]
    df["주요_불임_원인_합계"] = df[primary_cols].sum(axis=1)

    df["이식_효율"] = np.where(df["총 생성 배아 수"]>0, df["이식된 배아 수"]/df["총 생성 배아 수"], 0)
    if "미세주입에서 생성된 배아 수" in df.columns:
        df["ICSI_비율"] = np.where(df["총 생성 배아 수"]>0, df["미세주입에서 생성된 배아 수"]/df["총 생성 배아 수"], 0)
    df["과거_임신_성공률"] = np.where(df["총 시술 횟수"]>0, df["총 임신 횟수"]/(df["총 시술 횟수"]+1), 0)

    for col in ["동결 배아 사용 여부","신선 배아 사용 여부","기증 배아 사용 여부"]:
        if col not in df.columns: df[col] = 0
    df["배아_전략"] = df["동결 배아 사용 여부"]*1 + df["신선 배아 사용 여부"]*0 + df["기증 배아 사용 여부"]*2
    df["저장_비율"] = np.where(df["총 생성 배아 수"]>0, df["저장된 배아 수"]/df["총 생성 배아 수"], 0)
    df["기증_사용"] = ((df.get("난자 출처", pd.Series(0,index=df.index))==1)|(df.get("정자 출처", pd.Series(0,index=df.index))==1)).astype(int)

    if "해동된 배아 수" in df.columns:
        df["동결배아_활용률"]   = np.where(df["저장된 배아 수"]>0, df["해동된 배아 수"]/df["저장된 배아 수"], 0)
        df["ICSI동결_이식비율"] = np.where(df["이식된 배아 수"]>0, df["미세주입 후 저장된 배아 수"]/df["이식된 배아 수"], 0)
        df["순수동결_주기"]     = ((df["동결 배아 사용 여부"]==1)&(df["신선 배아 사용 여부"]==0)).astype(int)
        remaining               = (df["저장된 배아 수"]-df["해동된 배아 수"]).clip(lower=0)
        df["동결배아_잉여율"]   = np.where(df["저장된 배아 수"]>0, remaining/df["저장된 배아 수"], 0)
        df["동결배아_총량"]     = df["저장된 배아 수"]+df["해동된 배아 수"]

    if "수집된 신선 난자 수" in df.columns:
        age_num = df["시술 당시 나이"].replace(-1, np.nan)
        egg_cnt = df["수집된 신선 난자 수"]
        df["나이x난자수"]       = (age_num*egg_cnt).fillna(0)
        df["나이보정_난자효율"] = egg_cnt/(age_num.fillna(0)+1)
        df["고령저반응"] = ((age_num>=3)&(egg_cnt<egg_cnt.median())).fillna(False).astype(int)
        df["젊고고수확"] = ((age_num<=1)&(egg_cnt>=egg_cnt.quantile(0.75))).fillna(False).astype(int)

    df["배아_손실수"]       = (df["총 생성 배아 수"]-df["이식된 배아 수"]-df["저장된 배아 수"]).clip(lower=0)
    df["배아_총활용률"]     = np.where(df["총 생성 배아 수"]>0,(df["이식된 배아 수"]+df["저장된 배아 수"])/df["총 생성 배아 수"],0).clip(0,1)
    embryo_max              = max(df["총 생성 배아 수"].max()+1, 9)
    df["배아_풍요도"]       = pd.cut(df["총 생성 배아 수"],bins=[-1,0,3,7,embryo_max],labels=[0,1,2,3]).astype(float).fillna(0).astype(int)
    df["다배아이식_플래그"] = (df["이식된 배아 수"]>=3).astype(int)
    if "미세주입에서 생성된 배아 수" in df.columns:
        df["ICSI배아_우위비율"] = np.where(df["총 생성 배아 수"]>0, df["미세주입에서 생성된 배아 수"]/df["총 생성 배아 수"], 0)
    denom = df["이식된 배아 수"]+df["저장된 배아 수"]
    df["이식_집중도"] = np.where(denom>0, df["이식된 배아 수"]/denom, 0)

    if "시술 유형" in df.columns:
        age2 = df["시술 당시 나이"].replace(-1,0)
        df["나이x시술유형"]      = age2*df["시술 유형"]
        df["고령IVF"]            = ((df["시술 당시 나이"]>=3)&(df["시술 유형"]==1)).astype(int)
        df["젊은DI"]             = ((df["시술 당시 나이"]<=1)&(df["시술 유형"]==0)).astype(int)
        df["나이_시술_복합코드"] = df["시술 당시 나이"].clip(lower=0)*10+df["시술 유형"]

    if "총 시술 횟수" in df.columns:
        age3  = df["시술 당시 나이"].replace(-1,0)
        trial = df["총 시술 횟수"].replace(-1,0)
        df["나이x총시술횟수"] = age3*trial
        if "클리닉 내 총 시술 횟수" in df.columns:
            df["나이x클리닉시술횟수"] = age3*df["클리닉 내 총 시술 횟수"].replace(-1,0)
        if "IVF 시술 횟수" in df.columns:
            ivf = df["IVF 시술 횟수"].replace(-1,0)
            df["나이xIVF횟수"]  = age3*ivf
            df["시술부담_지수"] = age3*(trial+ivf)/2
        df["고령반복시술"] = ((df["시술 당시 나이"]>=3)&(trial>=3)).astype(int)
    return df


def add_embryo_quality_features(df):
    """기존 배아 품질 파생변수 (문서4 기반, 14개)"""
    out = df.copy()
    for c in ["총 생성 배아 수","수집된 신선 난자 수","미세주입에서 생성된 배아 수","미세주입된 난자 수","이식된 배아 수"]:
        if c in out.columns: out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0)
    if "배아 이식 경과일" in out.columns:
        out["배아 이식 경과일"] = pd.to_numeric(out["배아 이식 경과일"], errors="coerce")

    out["수정률"]      = np.where(out["수집된 신선 난자 수"]>0, out["총 생성 배아 수"]/out["수집된 신선 난자 수"], np.nan).clip(0,1)
    out["수정률_결측"] = (out["수집된 신선 난자 수"]==0).astype(int)
    out["수정률"]      = out["수정률"].fillna(0)
    out["수정률_양호"] = (out["수정률"]>=0.5).astype(int)
    out["수정률_구간"] = pd.cut(out["수정률"],bins=[-0.01,0.3,0.5,0.7,1.01],labels=[0,1,2,3]).astype(float)

    out["is_D5"]   = np.where(out["배아 이식 경과일"].isnull(), np.nan, (out["배아 이식 경과일"]>=5).astype(float))
    out["D5_결측"]  = out["배아 이식 경과일"].isnull().astype(int)
    out["is_D5"]   = out["is_D5"].fillna(0)
    out["이식단계"] = np.where(out["배아 이식 경과일"].isnull(),-1,
                               np.where(out["배아 이식 경과일"]<=3,0,
                               np.where(out["배아 이식 경과일"]>=5,1,0)))

    out["ICSI수정률"]      = np.where(out["미세주입된 난자 수"]>0, out["미세주입에서 생성된 배아 수"]/out["미세주입된 난자 수"], np.nan).clip(0,1)
    out["ICSI_미시행"]     = (out["미세주입된 난자 수"]==0).astype(int)
    out["ICSI수정률"]      = out["ICSI수정률"].fillna(0)
    out["ICSI수정률_양호"] = (out["ICSI수정률"]>=0.7).astype(int)
    out["ICSI수정률_구간"] = pd.cut(out["ICSI수정률"],bins=[-0.01,0.5,0.7,0.9,1.01],labels=[0,1,2,3]).astype(float)
    out["이식수_최적"]  = out["이식된 배아 수"].isin([1.0,2.0]).astype(int)
    out["최적이식_D5"] = out["이식수_최적"]*out["is_D5"]
    out["배아질_복합"] = out["수정률"].clip(0,1)*0.4 + out["is_D5"]*0.4 + out["이식수_최적"]*0.2
    return out


def add_new_features(df):
    """★ 신규 파생변수 29개 (행 단위 연산 → Leakage 없음)"""
    out = df.copy()
    for c in ["총 생성 배아 수","이식된 배아 수","저장된 배아 수","수집된 신선 난자 수",
              "저장된 신선 난자 수","혼합된 난자 수","해동된 배아 수","파트너 정자와 혼합된 난자 수",
              "배아 이식 경과일","난자 혼합 경과일","시술 당시 나이","시술 유형",
              "총 시술 횟수","클리닉 내 총 시술 횟수","IVF 시술 횟수",
              "총 임신 횟수","총 출산 횟수","IVF 임신 횟수","IVF 출산 횟수",
              "DI 임신 횟수","DI 출산 횟수","미세주입된 난자 수","미세주입에서 생성된 배아 수"]:
        if c in out.columns: out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0)

    # 그룹 A. 배아 이식 경과일 관련 (corr 0.208~0.252)
    out["혼합_이식_간격"]   = (out["배아 이식 경과일"]-out["난자 혼합 경과일"]).fillna(0)   # corr=0.252
    out["이식일_D5이상"]    = (out["배아 이식 경과일"]>=5).fillna(False).astype(int)          # corr=0.228
    out["이식일x이식수"]    = out["배아 이식 경과일"].fillna(0)*out["이식된 배아 수"].fillna(0) # corr=0.208
    out["이식일_결측"]      = (out["배아 이식 경과일"]==0).astype(int)                        # corr=-0.243
    out["정확히_D5"]        = (out["배아 이식 경과일"]==5.0).astype(int)                      # corr=0.228
    out["D5_최적이식_복합"] = ((out["배아 이식 경과일"]>=5)&(out["이식된 배아 수"].isin([1.0,2.0]))).fillna(False).astype(int)

    # 그룹 B. 배아/난자 품질 강화
    fert_rate = np.where(out["수집된 신선 난자 수"]>0, out["총 생성 배아 수"]/out["수집된 신선 난자 수"],0).clip(0,1)
    out["생성배아_품질복합"] = out["총 생성 배아 수"]*fert_rate                               # corr=0.126
    out["난자_풍부도"]       = pd.cut(out["수집된 신선 난자 수"],bins=[-1,0,4,8,12,999],labels=[0,1,2,3,4]).astype(float).fillna(0).astype(int)  # corr=0.107
    out["신선난자_저장비율"] = np.where(out["수집된 신선 난자 수"]>0, out["저장된 신선 난자 수"]/out["수집된 신선 난자 수"],0)  # corr=-0.056
    out["파트너정자_활용률"] = np.where(out["혼합된 난자 수"]>0, out["파트너 정자와 혼합된 난자 수"]/out["혼합된 난자 수"],0)
    out["전체_배아_효율"]    = np.where(out["수집된 신선 난자 수"]>0,(out["이식된 배아 수"]+out["저장된 배아 수"])/out["수집된 신선 난자 수"],0).clip(0,5)

    # 그룹 C. 시술 이력 강화
    out["초회시술"]         = (out["총 시술 횟수"]==0).astype(int)                            # corr=0.059
    out["초회IVF"]          = ((out["IVF 시술 횟수"]==0)&(out["총 시술 횟수"]<=1)).astype(int)
    out["클리닉_집중도"]    = np.where(out["총 시술 횟수"]>0, out["클리닉 내 총 시술 횟수"]/out["총 시술 횟수"],1.0).clip(0,1)
    out["IVF_임신_전환율"]  = np.where(out["IVF 시술 횟수"]>0, out["IVF 임신 횟수"]/out["IVF 시술 횟수"],0).clip(0,1)
    out["임신_출산_전환율"] = np.where(out["총 임신 횟수"]>0, out["총 출산 횟수"]/out["총 임신 횟수"],0).clip(0,1)
    out["출산_경험"]        = (out["총 출산 횟수"]>0).astype(int)
    out["IVF_출산_경험"]    = (out["IVF 출산 횟수"]>0).astype(int)

    # 그룹 D. 불임 원인 세분화
    male_cols   = [c for c in ["남성 주 불임 원인","남성 부 불임 원인","불임 원인 - 남성 요인",
                               "불임 원인 - 정자 농도","불임 원인 - 정자 운동성","불임 원인 - 정자 형태",
                               "불임 원인 - 정자 면역학적 요인"] if c in out.columns]
    female_cols = [c for c in ["여성 주 불임 원인","여성 부 불임 원인","불임 원인 - 난관 질환",
                               "불임 원인 - 배란 장애","불임 원인 - 자궁내막증","불임 원인 - 자궁경부 문제"] if c in out.columns]
    out["남성불임_복합"]    = out[male_cols].fillna(0).sum(axis=1)                             # corr=0.026
    out["여성불임_복합"]    = out[female_cols].fillna(0).sum(axis=1)
    out["복합_불임_여부"]   = ((out["남성불임_복합"]>0)&(out["여성불임_복합"]>0)).astype(int)
    cause_all = [c for c in out.columns if "불임 원인 -" in c]
    age_num = out["시술 당시 나이"].replace(-1,0)
    out["나이x불임원인수"] = age_num*out[cause_all].fillna(0).sum(axis=1)                      # corr=-0.077

    # 그룹 E. 나이 교호작용 강화
    out["나이xD5"]         = age_num*out["이식일_D5이상"]                                      # corr=0.064
    out["나이x수정률"]     = age_num*fert_rate
    out["나이x클리닉횟수"] = age_num*out["클리닉 내 총 시술 횟수"].replace(-1,0)
    return out


## 4. 전체 전처리 파이프라인 & 데이터 준비

In [6]:
def full_pipeline(df, medians=None, fit=False):
    df = expand_reason(df.copy())
    df = expand_procedure(df)
    df.drop(columns=["배아 생성 주요 이유","특정 시술 유형"], inplace=True, errors="ignore")
    df, medians = preprocess(df, medians=medians, fit=fit)
    df = feature_engineering(df)
    df = add_embryo_quality_features(df)
    df = add_new_features(df)
    return df, medians

X_raw      = train.drop(columns=[ID_COL, TARGET], errors="ignore")
y          = train[TARGET]
X_test_raw = test.drop(columns=[ID_COL], errors="ignore")

# ★ train에서만 fit → test에는 train 통계 적용 (Leakage 방지)
X_pp, train_medians = full_pipeline(X_raw, fit=True)
X_test_pp, _        = full_pipeline(X_test_raw, medians=train_medians, fit=False)

common_cols = [c for c in X_pp.columns if c in X_test_pp.columns]
X_pp        = X_pp[common_cols]
X_test_pp   = X_test_pp[common_cols]

print(f"Train: {X_pp.shape} | Test: {X_test_pp.shape}")
print(f"잔여 결측치(Train): {X_pp.isnull().sum().sum()}")
print(f"총 피처 수: {X_pp.shape[1]}개  (기존 46 + 신규 29)")

X_arr      = X_pp.values.astype(np.float32)
y_arr      = y.values
X_test_arr = X_test_pp.values.astype(np.float32)
skf        = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)


Train: (256351, 147) | Test: (90067, 147)
잔여 결측치(Train): 0
총 피처 수: 147개  (기존 46 + 신규 29)


## Phase 1 — XGB × CatBoost Optuna 튜닝 (각 30회)

**전략:** OOF 기반 AUC를 목적함수로 Optuna 베이즈 최적화. test 정보는 전혀 사용하지 않음.


In [7]:
# ── Phase 1-A: XGBoost Optuna ──────────────────────────────────────────────
print("=== Phase 1-A: XGBoost Optuna 탐색 ===")

def xgb_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int("n_estimators", 500, 3000, step=100),
        learning_rate     = trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        max_depth         = trial.suggest_int("max_depth", 4, 9),
        subsample         = trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree  = trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha         = trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        reg_lambda        = trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        min_child_weight  = trial.suggest_int("min_child_weight", 1, 20),
        gamma             = trial.suggest_float("gamma", 0, 5),
        scale_pos_weight  = trial.suggest_float("scale_pos_weight", 1.5, 4.0),
        tree_method       = "hist",
        eval_metric       = "auc",
        early_stopping_rounds = 50,
        random_state      = RANDOM_STATE,
        n_jobs=-1, verbosity=0,
    )
    oof = np.zeros(len(X_arr))
    for tr, va in skf.split(X_arr, y_arr):
        m = xgb.XGBClassifier(**params)
        m.fit(X_arr[tr], y_arr[tr], eval_set=[(X_arr[va],y_arr[va])], verbose=False)
        oof[va] = m.predict_proba(X_arr[va])[:,1]
    return roc_auc_score(y_arr, oof)

study_xgb = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_xgb.optimize(xgb_objective, n_trials=30, show_progress_bar=True)
best_xgb_params = study_xgb.best_params
print(f"XGB 최적 AUC: {study_xgb.best_value:.5f}")
print(f"XGB 최적 파라미터: {best_xgb_params}")


=== Phase 1-A: XGBoost Optuna 탐색 ===


  0%|          | 0/30 [00:00<?, ?it/s]

XGB 최적 AUC: 0.73983
XGB 최적 파라미터: {'n_estimators': 2100, 'learning_rate': 0.010210623940780495, 'max_depth': 6, 'subsample': 0.7481946890636312, 'colsample_bytree': 0.7363570033289744, 'reg_alpha': 6.956365119888485, 'reg_lambda': 9.200285059730406, 'min_child_weight': 7, 'gamma': 4.743047347372708, 'scale_pos_weight': 1.5089588921620083}


In [8]:
# ── Phase 1-B: CatBoost Optuna ─────────────────────────────────────────────
print("=== Phase 1-B: CatBoost Optuna 탐색 ===")

def cat_objective(trial):
    params = dict(
        iterations        = trial.suggest_int("iterations", 500, 3000, step=100),
        learning_rate     = trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        depth             = trial.suggest_int("depth", 4, 10),
        l2_leaf_reg       = trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        bagging_temperature = trial.suggest_float("bagging_temperature", 0, 2),
        random_strength   = trial.suggest_float("random_strength", 0, 2),
        border_count      = trial.suggest_int("border_count", 32, 255),
        auto_class_weights = "Balanced",
        eval_metric       = "AUC",
        early_stopping_rounds = 50,
        random_seed       = RANDOM_STATE,
        verbose           = 0,
    )
    oof = np.zeros(len(X_arr))
    for tr, va in skf.split(X_arr, y_arr):
        m = CatBoostClassifier(**params)
        m.fit(X_arr[tr], y_arr[tr], eval_set=(X_arr[va],y_arr[va]), use_best_model=True)
        oof[va] = m.predict_proba(X_arr[va])[:,1]
    return roc_auc_score(y_arr, oof)

study_cat = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_cat.optimize(cat_objective, n_trials=30, show_progress_bar=True)
best_cat_params = study_cat.best_params
print(f"Cat 최적 AUC: {study_cat.best_value:.5f}")
print(f"Cat 최적 파라미터: {best_cat_params}")


=== Phase 1-B: CatBoost Optuna 탐색 ===


  0%|          | 0/30 [00:00<?, ?it/s]

Cat 최적 AUC: 0.73974
Cat 최적 파라미터: {'iterations': 2400, 'learning_rate': 0.04562257814760613, 'depth': 6, 'l2_leaf_reg': 4.430711373313008, 'bagging_temperature': 1.302241043962474, 'random_strength': 0.8885571659165084, 'border_count': 58}


In [9]:
# ── Phase 1-C: 최적 파라미터로 OOF + Test 예측 ────────────────────────────
print("=== Phase 1-C: 최적 파라미터 전체 학습 ===")

oof_xgb_p1  = np.zeros(len(X_arr))
pred_xgb_p1 = np.zeros(len(X_test_arr))
oof_cat_p1  = np.zeros(len(X_arr))
pred_cat_p1 = np.zeros(len(X_test_arr))

xgb_final_params = {**best_xgb_params,
    "tree_method":"hist","eval_metric":"auc",
    "early_stopping_rounds":100,"random_state":RANDOM_STATE,"n_jobs":-1,"verbosity":0}
cat_final_params = {**best_cat_params,
    "auto_class_weights":"Balanced","eval_metric":"AUC",
    "early_stopping_rounds":100,"random_seed":RANDOM_STATE,"verbose":0}

for fold,(tr,va) in enumerate(skf.split(X_arr,y_arr),1):
    # XGB
    mx = xgb.XGBClassifier(**xgb_final_params)
    mx.fit(X_arr[tr],y_arr[tr],eval_set=[(X_arr[va],y_arr[va])],verbose=False)
    oof_xgb_p1[va]  = mx.predict_proba(X_arr[va])[:,1]
    pred_xgb_p1    += mx.predict_proba(X_test_arr)[:,1]/N_SPLITS
    # Cat
    mc = CatBoostClassifier(**cat_final_params)
    mc.fit(X_arr[tr],y_arr[tr],eval_set=(X_arr[va],y_arr[va]),use_best_model=True)
    oof_cat_p1[va]  = mc.predict_proba(X_arr[va])[:,1]
    pred_cat_p1    += mc.predict_proba(X_test_arr)[:,1]/N_SPLITS
    print(f"  Fold {fold} | XGB={roc_auc_score(y_arr[va],oof_xgb_p1[va]):.5f}  CAT={roc_auc_score(y_arr[va],oof_cat_p1[va]):.5f}")

print(f"\nPhase1 OOF | XGB={roc_auc_score(y_arr,oof_xgb_p1):.5f}  CAT={roc_auc_score(y_arr,oof_cat_p1):.5f}")

# XGB × Cat 가중치 최적화
def neg_auc_p1(w):
    w = np.array(w); w = w/w.sum()
    return -roc_auc_score(y_arr, w[0]*oof_xgb_p1 + w[1]*oof_cat_p1)
res_p1 = minimize(neg_auc_p1, x0=[0.5,0.5], bounds=[(0,1)]*2, method="L-BFGS-B")
w_p1   = np.array(res_p1.x)/np.array(res_p1.x).sum()
oof_blend_p1  = w_p1[0]*oof_xgb_p1 + w_p1[1]*oof_cat_p1
pred_blend_p1 = w_p1[0]*pred_xgb_p1 + w_p1[1]*pred_cat_p1
print(f"Phase1 XGB×Cat 앙상블 AUC: {roc_auc_score(y_arr,oof_blend_p1):.5f}  (w_xgb={w_p1[0]:.3f}, w_cat={w_p1[1]:.3f})")

# Phase 1 제출 저장
sub_p1 = sub.copy(); sub_p1["probability"] = pred_blend_p1
sub_p1.to_csv("../submission_phase1.csv", index=False)
print("✅ submission_phase1.csv 저장 완료")


=== Phase 1-C: 최적 파라미터 전체 학습 ===
  Fold 1 | XGB=0.73757  CAT=0.73762
  Fold 2 | XGB=0.74242  CAT=0.74297
  Fold 3 | XGB=0.74004  CAT=0.73972
  Fold 4 | XGB=0.73837  CAT=0.73790
  Fold 5 | XGB=0.74100  CAT=0.74070

Phase1 OOF | XGB=0.73987  CAT=0.73978
Phase1 XGB×Cat 앙상블 AUC: 0.74007  (w_xgb=0.500, w_cat=0.500)
✅ submission_phase1.csv 저장 완료


## Phase 2 — 트리 다양성 확장

**전략:** LGB (dart 모드) + XGB (두 버전) + Cat + Seed 앙상블로 다양성 극대화


In [10]:
print("=== Phase 2: 트리 다양성 확장 ===")

# LGB Optuna
def lgb_objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int("n_estimators", 500, 3000, step=100),
        learning_rate    = trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        num_leaves       = trial.suggest_int("num_leaves", 31, 255),
        max_depth        = trial.suggest_int("max_depth", 4, 12),
        subsample        = trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha        = trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        reg_lambda       = trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        min_child_samples = trial.suggest_int("min_child_samples", 5, 100),
        boosting_type    = trial.suggest_categorical("boosting_type", ["gbdt","goss"]),  # ← 원래 ["gbdt","dart","goss"] 였던 부분
        class_weight     = "balanced",
        random_state     = RANDOM_STATE,
        n_jobs=-1, verbose=-1,
    )
    oof = np.zeros(len(X_arr))
    for tr, va in skf.split(X_arr, y_arr):
        m = lgb.LGBMClassifier(**params)
        m.fit(X_arr[tr],y_arr[tr],eval_set=[(X_arr[va],y_arr[va])],
              callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(False)])
        oof[va] = m.predict_proba(X_arr[va])[:,1]
    return roc_auc_score(y_arr, oof)

study_lgb = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_lgb.optimize(lgb_objective, n_trials=30, show_progress_bar=True)
best_lgb_params = study_lgb.best_params
print(f"LGB 최적 AUC: {study_lgb.best_value:.5f}")
print(f"LGB 최적 파라미터: {best_lgb_params}")

=== Phase 2: 트리 다양성 확장 ===


  0%|          | 0/30 [00:00<?, ?it/s]

LGB 최적 AUC: 0.73924
LGB 최적 파라미터: {'n_estimators': 1300, 'learning_rate': 0.020263188298172464, 'num_leaves': 150, 'max_depth': 4, 'subsample': 0.91206952631544, 'colsample_bytree': 0.9428380005720536, 'reg_alpha': 0.9061803492912888, 'reg_lambda': 3.7279273338107584, 'min_child_samples': 100, 'boosting_type': 'gbdt'}


In [11]:
# Seed 앙상블 (XGB, Cat, LGB 각 3 seed)
SEEDS = [42, 2024, 777]

oof_models_p2  = {}
pred_models_p2 = {}

model_configs = {
    "XGB_s42" : ("xgb", {**xgb_final_params, "random_state":42}),
    "XGB_s2024": ("xgb", {**xgb_final_params, "random_state":2024}),
    "XGB_s777" : ("xgb", {**xgb_final_params, "random_state":777}),
    "CAT_s42"  : ("cat", {**cat_final_params, "random_seed":42}),
    "CAT_s2024": ("cat", {**cat_final_params, "random_seed":2024}),
    "CAT_s777" : ("cat", {**cat_final_params, "random_seed":777}),
    "LGB_s42"  : ("lgb", {**best_lgb_params,"class_weight":"balanced","random_state":42,"n_jobs":-1,"verbose":-1}),
    "LGB_s2024": ("lgb", {**best_lgb_params,"class_weight":"balanced","random_state":2024,"n_jobs":-1,"verbose":-1}),
    "LGB_s777" : ("lgb", {**best_lgb_params,"class_weight":"balanced","random_state":777,"n_jobs":-1,"verbose":-1}),
}

for name,(mtype,params) in model_configs.items():
    oof  = np.zeros(len(X_arr))
    pred = np.zeros(len(X_test_arr))
    for tr,va in skf.split(X_arr,y_arr):
        if mtype=="xgb":
            m = xgb.XGBClassifier(**params)
            m.fit(X_arr[tr],y_arr[tr],eval_set=[(X_arr[va],y_arr[va])],verbose=False)
        elif mtype=="cat":
            m = CatBoostClassifier(**params)
            m.fit(X_arr[tr],y_arr[tr],eval_set=(X_arr[va],y_arr[va]),use_best_model=True)
        else:
            m = lgb.LGBMClassifier(**params)
            m.fit(X_arr[tr],y_arr[tr],eval_set=[(X_arr[va],y_arr[va])],
                  callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(False)])
        oof[va] = m.predict_proba(X_arr[va])[:,1]
        pred   += m.predict_proba(X_test_arr)[:,1]/N_SPLITS
    auc = roc_auc_score(y_arr, oof)
    oof_models_p2[name]  = oof
    pred_models_p2[name] = pred
    print(f"  {name:<12} OOF AUC: {auc:.5f}")

# Phase 2 앙상블 (단순 평균)
oof_blend_p2  = np.mean(list(oof_models_p2.values()), axis=0)
pred_blend_p2 = np.mean(list(pred_models_p2.values()), axis=0)
print(f"\nPhase2 앙상블 OOF AUC: {roc_auc_score(y_arr,oof_blend_p2):.5f}")

sub_p2 = sub.copy(); sub_p2["probability"] = pred_blend_p2
sub_p2.to_csv("submission_phase2.csv", index=False)
print("✅ submission_phase2.csv 저장 완료")


  XGB_s42      OOF AUC: 0.73987
  XGB_s2024    OOF AUC: 0.73982
  XGB_s777     OOF AUC: 0.73981
  CAT_s42      OOF AUC: 0.73978
  CAT_s2024    OOF AUC: 0.73977
  CAT_s777     OOF AUC: 0.73969
  LGB_s42      OOF AUC: 0.73924
  LGB_s2024    OOF AUC: 0.73915
  LGB_s777     OOF AUC: 0.73912

Phase2 앙상블 OOF AUC: 0.73998
✅ submission_phase2.csv 저장 완료
